# Generate completions using preselected best-on-contour occluders

This notebook keeps the original generation logic, but changes the occluder source.

## New input logic
For each case:
- the **silhouette**, `baseGrid`, and class information come from the base case JSONL
- the **occluder** comes from the precomputed CSV of best intersecting occluders

The new occluder is then used to:
- render the occluded image
- recompute contour intersections
- derive `start_pt`, `end_pt`, and the hidden fraction
- generate completions with the same downstream pipeline

In [1]:
from pathlib import Path
import json
import re
import shutil
from concurrent.futures import ProcessPoolExecutor, as_completed
from collections import Counter

import numpy as np
import pandas as pd

In [2]:
ROOT = Path("..").resolve()

BEST_OCCLUDER_CSV = "/home/hschatzle/data/storage-occ-v2/repos/monte-carlo-selection/notebooks/part1_outputs/intersecting_occluders/tables/best_intersecting_occluder_by_case_model_size.csv"

CASESET_NAME = "occlusion"   # or "sweep", "contrasts"
CASESET_DIR = ROOT / "data" / "cases" / CASESET_NAME

MASTER_LIB = ROOT / "data" / "master" / "master_lib.mat"

OUT_PARENT = ROOT / "data" / "cases" / "generated_from_best_occluders"

H, W = 224, 224
MARGIN = 5
SUPERSAMPLE = 4

N_WORKERS = 60
TOTAL_ATTEMPTS = 10_000

SAMPLES_MODE = "matlab_100"

REFIT_N_CTRL = 10
REFIT_SUBDIV = 18
REFIT_JITTER = 0.008
REFIT_MAX_ATTEMPTS = 12

FLUSH = 1000

CASE_FRACTION = 0.22
CASE_SHRINK_GAMMA = 0.96
CASE_MAX_SHRINK_ITERS = 30
CASE_SMOOTH_WIN = 1

In [3]:
best_df = pd.read_csv(BEST_OCCLUDER_CSV)

print("Loaded best occluder table:", best_df.shape)
display(best_df.head())

Loaded best occluder table: (320, 19)


,case_id,model_type,occluder_size,stride,x0,y0,x_center,y_center,x0_norm,y0_norm,x1_norm,y1_norm,x_center_norm,y_center_norm,baseline_target_logit,post_occlusion_target_logit,target_logit_drop,intersects_shape,source_file
0,ns_cow_202,resnet50_geirhos_tl,8,4,124,108,128.0,112.0,0.553571,0.482143,0.589286,0.517857,0.571429,0.500000,7.092349,6.706945,0.385403,1,/home/hschatzle/data/storage-occ-v2/repos/mont...
1,ns_cow_202,resnet50_geirhos_tl,16,8,192,96,200.0,104.0,0.857143,0.428571,0.928571,0.500000,0.892857,0.464286,7.092349,6.336758,0.755591,1,/home/hschatzle/data/storage-occ-v2/repos/mont...
2,ns_cow_202,resnet50_geirhos_tl,32,16,192,96,208.0,112.0,0.857143,0.428571,1.000000,0.571429,0.928571,0.500000,7.092349,6.050385,1.041964,1,/home/hschatzle/data/storage-occ-v2/repos/mont...
3,ns_cow_202,resnet50_geirhos_tl,64,32,96,96,128.0,128.0,0.428571,0.428571,0.714286,0.714286,0.571429,0.571429,7.092349,3.778350,3.313999,1,/home/hschatzle/data/storage-occ-v2/repos/mont...
4,ns_cow_202,resnet50_tl_20250829,8,4,60,176,64.0,180.0,0.267857,0.785714,0.303571,0.821429,0.285714,0.803571,150.310638,149.334610,0.976028,1,/home/hschatzle/data/storage-occ-v2/repos/mont...


In [4]:
# Optional filtering.
# Example:
# KEEP_MODELS = ["resnet50_geirhos_tl"]
# KEEP_CASE_IDS = ["ns_cow_202"]

KEEP_MODELS = None
KEEP_CASE_IDS = None
KEEP_OCCLUDER_SIZES = [8]

run_df = best_df.copy()

if KEEP_MODELS is not None:
    run_df = run_df[run_df["model_type"].isin(KEEP_MODELS)].copy()

if KEEP_CASE_IDS is not None:
    run_df = run_df[run_df["case_id"].isin(KEEP_CASE_IDS)].copy()

if KEEP_OCCLUDER_SIZES is not None:
    run_df = run_df[run_df["occluder_size"].isin(KEEP_OCCLUDER_SIZES)].copy()

run_df = run_df.sort_values(["case_id", "model_type", "occluder_size"]).reset_index(drop=True)

print("Rows to run:", len(run_df))
display(run_df.head(20))

Rows to run: 80


,case_id,model_type,occluder_size,stride,x0,y0,x_center,y_center,x0_norm,y0_norm,x1_norm,y1_norm,x_center_norm,y_center_norm,baseline_target_logit,post_occlusion_target_logit,target_logit_drop,intersects_shape,source_file
0,ns_cow_202,resnet50_geirhos_tl,8,4,124,108,128.0,112.0,0.553571,0.482143,0.589286,0.517857,0.571429,0.500000,7.092349,6.706945,0.385403,1,/home/hschatzle/data/storage-occ-v2/repos/mont...
1,ns_cow_202,resnet50_tl_20250829,8,4,60,176,64.0,180.0,0.267857,0.785714,0.303571,0.821429,0.285714,0.803571,150.310638,149.334610,0.976028,1,/home/hschatzle/data/storage-occ-v2/repos/mont...
2,ns_cow_205,resnet50_geirhos_tl,8,4,72,40,76.0,44.0,0.321429,0.178571,0.357143,0.214286,0.339286,0.196429,4.972391,4.603109,0.369282,1,/home/hschatzle/data/storage-occ-v2/repos/mont...
3,ns_cow_205,resnet50_tl_20250829,8,4,92,176,96.0,180.0,0.410714,0.785714,0.446429,0.821429,0.428571,0.803571,142.628540,141.270203,1.358337,1,/home/hschatzle/data/storage-occ-v2/repos/mont...
4,ns_cow_350,resnet50_geirhos_tl,8,4,8,56,12.0,60.0,0.035714,0.250000,0.071429,0.285714,0.053571,0.267857,5.908604,5.489727,0.418877,1,/home/hschatzle/data/storage-occ-v2/repos/mont...
5,ns_cow_350,resnet50_tl_20250829,8,4,48,48,52.0,52.0,0.214286,0.214286,0.250000,0.250000,0.232143,0.232143,132.863724,132.048996,0.814728,1,/home/hschatzle/data/storage-occ-v2/repos/mont...
6,ns_cow_510,resnet50_geirhos_tl,8,4,20,76,24.0,80.0,0.089286,0.339286,0.125000,0.375000,0.107143,0.357143,6.473172,6.123373,0.349799,1,/home/hschatzle/data/storage-occ-v2/repos/mont...
7,ns_cow_510,resnet50_tl_20250829,8,4,48,48,52.0,52.0,0.214286,0.214286,0.250000,0.250000,0.232143,0.232143,152.570129,151.577057,0.993073,1,/home/hschatzle/data/storage-occ-v2/repos/mont...
8,ns_cow_780,resnet50_geirhos_tl,8,4,156,36,160.0,40.0,0.696429,0.160714,0.732143,0.196429,0.714286,0.178571,6.982400,6.653270,0.329130,1,/home/hschatzle/data/storage-occ-v2/repos/mont...
9,ns_cow_780,resnet50_tl_20250829,8,4,156,172,160.0,176.0,0.696429,0.767857,0.732143,0.803571,0.714286,0.785714,143.739960,142.898972,0.840988,1,/home/hschatzle/data/storage-occ-v2/repos/mont...


In [5]:
import sys
sys.path.append(str(ROOT / "scripts"))

from shape_gen.io_mat import unit_to_pixel
from shape_gen.library import load_master_records, build_class_index
from shape_gen.generate_parallel3 import generate_completions, save_metadata_jsonl
from shape_gen.heatmap.xy_store import save_xy_npz
from shape_gen.geom_bbox import compute_bbox
from shape_gen.render import draw_and_save
from shape_gen.intersections2 import find_intersection_points_multiple

In [6]:
records = load_master_records(MASTER_LIB)
classes, byClass = build_class_index(records)

print("Loaded records:", len(records))
print("Classes:", len(classes))

Loaded records: 54000
Classes: 54


In [7]:
def find_case_jsonl(case_id: str, caseset_dir: Path) -> Path:
    case_dir = caseset_dir / case_id / "generated"
    preferred = case_dir / f"{case_id}.jsonl"
    if preferred.exists():
        return preferred

    candidates = sorted(case_dir.glob("*.jsonl"))
    if not candidates:
        raise FileNotFoundError(f"No JSONL found for case_id={case_id} in {case_dir}")
    return candidates[0]

In [8]:
def load_case_from_jsonl(jsonl_path: Path, default_baseGrid: int = 256):
    rows = [json.loads(line) for line in jsonl_path.read_text().splitlines() if line.strip()]
    row = rows[-1]

    if "silhouette_u" in row:
        sil_u = np.asarray(row["silhouette_u"], float)
        baseGrid = int(row["baseGrid"])
        sil_class = row.get("sil_class", None)
    else:
        sil_u = np.asarray(row["shape_contour_xy"], float)
        baseGrid = int(row.get("baseGrid", default_baseGrid))
        sil_class = row.get("sil_class", row.get("category", None))

    return sil_u, baseGrid, sil_class, row

In [9]:
def rect_poly_from_csv_row(row, baseGrid: int):
    """
    Build the occluder rectangle in unit coordinates, then convert to pixel coordinates.

    Priority:
    1. use normalized coordinates already saved in the CSV
    2. otherwise reconstruct from x0, y0, occluder_size assuming the CSV was based on a 224x224 grid
    """
    if all(col in row.index for col in ["x0_norm", "y0_norm", "x1_norm", "y1_norm"]):
        x0u = float(row["x0_norm"])
        y0u = float(row["y0_norm"])
        x1u = float(row["x1_norm"])
        y1u = float(row["y1_norm"])
    else:
        image_w = 224.0
        image_h = 224.0
        x0u = float(row["x0"]) / image_w
        y0u = float(row["y0"]) / image_h
        x1u = (float(row["x0"]) + float(row["occluder_size"])) / image_w
        y1u = (float(row["y0"]) + float(row["occluder_size"])) / image_h

    rect_u = np.array([
        [x0u, y0u],
        [x1u, y0u],
        [x1u, y1u],
        [x0u, y1u],
    ], dtype=float)

    rect_px = unit_to_pixel(rect_u, baseGrid)
    return rect_u, rect_px

In [10]:
def _point_in_poly(point, poly):
    x, y = float(point[0]), float(point[1])
    inside = False
    n = len(poly)
    j = n - 1
    for i in range(n):
        xi, yi = float(poly[i, 0]), float(poly[i, 1])
        xj, yj = float(poly[j, 0]), float(poly[j, 1])
        hit = ((yi > y) != (yj > y)) and (
            x < (xj - xi) * (y - yi) / ((yj - yi) + 1e-12) + xi
        )
        if hit:
            inside = not inside
        j = i
    return inside


def _closed_seg_lengths(poly):
    nxt = np.roll(poly, -1, axis=0)
    return np.linalg.norm(nxt - poly, axis=1)


def _cum_closed_lengths(poly):
    seg = _closed_seg_lengths(poly)
    cum = np.concatenate([[0.0], np.cumsum(seg)])
    total = float(seg.sum())
    return seg, cum, total


def _project_point_to_closed_poly(poly, p):
    a = poly
    b = np.roll(poly, -1, axis=0)
    ab = b - a
    ap = p[None, :] - a

    denom = np.sum(ab * ab, axis=1)
    denom = np.where(denom < 1e-12, 1e-12, denom)

    t = np.sum(ap * ab, axis=1) / denom
    t = np.clip(t, 0.0, 1.0)

    proj = a + t[:, None] * ab
    d2 = np.sum((proj - p[None, :]) ** 2, axis=1)

    idx = int(np.argmin(d2))
    return idx, float(t[idx]), proj[idx]


def _s_of_projection(poly, idx, t):
    seg, cum, total = _cum_closed_lengths(poly)
    return (cum[idx] + t * seg[idx]) % total


def _point_at_s(poly, s):
    seg, cum, total = _cum_closed_lengths(poly)
    s = s % total

    idx = np.searchsorted(cum, s, side="right") - 1
    idx = min(max(idx, 0), len(seg) - 1)

    ds = s - cum[idx]
    t = 0.0 if seg[idx] < 1e-12 else ds / seg[idx]

    a = poly[idx]
    b = poly[(idx + 1) % len(poly)]
    return a + t * (b - a)


def _forward_arc_len(total, s0, s1):
    return (s1 - s0) % total


def choose_gap_pair_and_fraction(silhouette, occluder, pts):
    if pts.shape[0] < 2:
        raise RuntimeError(f"Need at least 2 intersections, got {pts.shape[0]}")

    silhouette = np.asarray(silhouette, dtype=float)
    occluder = np.asarray(occluder, dtype=float)
    pts = np.asarray(pts, dtype=float)

    _, _, total = _cum_closed_lengths(silhouette)

    items = []
    for p in pts:
        idx, t, proj = _project_point_to_closed_poly(silhouette, p)
        s = _s_of_projection(silhouette, idx, t)
        items.append((s, proj))

    items.sort(key=lambda x: x[0])

    if len(items) == 2:
        s0, p0 = items[0]
        s1, p1 = items[1]

        fwd = _forward_arc_len(total, s0, s1)
        bwd = total - fwd

        mid_fwd = _point_at_s(silhouette, s0 + 0.5 * fwd)
        hidden_is_fwd = _point_in_poly(mid_fwd, occluder)

        hidden_len = fwd if hidden_is_fwd else bwd
        hidden_fraction = float(np.clip(hidden_len / max(total, 1e-12), 1e-3, 0.95))

        return np.asarray(p0), np.asarray(p1), hidden_fraction

    best = None
    m = len(items)

    for i in range(m):
        s0, p0 = items[i]
        s1, p1 = items[(i + 1) % m]

        arc_len = _forward_arc_len(total, s0, s1)
        mid = _point_at_s(silhouette, s0 + 0.5 * arc_len)
        is_hidden = _point_in_poly(mid, occluder)

        if is_hidden:
            if best is None or arc_len > best[0]:
                best = (arc_len, p0, p1)

    if best is None:
        best_arc = -1.0
        best_pair = None
        for i in range(m):
            s0, p0 = items[i]
            s1, p1 = items[(i + 1) % m]
            arc_len = _forward_arc_len(total, s0, s1)
            if arc_len > best_arc:
                best_arc = arc_len
                best_pair = (p0, p1)

        hidden_len = best_arc
        p0, p1 = best_pair
    else:
        hidden_len, p0, p1 = best

    hidden_fraction = float(np.clip(hidden_len / max(total, 1e-12), 1e-3, 0.95))
    return np.asarray(p0), np.asarray(p1), hidden_fraction

In [11]:
def _worker_run(args):
    wid = int(args["worker_id"])
    out_dir = Path(args["out_dir"])
    tmp_dir = Path(args["tmp_dir"])
    tmp_dir.mkdir(parents=True, exist_ok=True)

    rng = np.random.default_rng(int(args["seed"]))

    metas, out_files_xy, polygons_xy, fail_counts = generate_completions(
        silhouette=args["silhouette"],
        occluder=args["occluder"],
        start_pt=args["start_pt"],
        end_pt=args["end_pt"],
        minX=int(args["minX"]),
        minY=int(args["minY"]),
        wBB=int(args["wBB"]),
        hBB=int(args["hBB"]),
        out_w=int(args["out_w"]),
        out_h=int(args["out_h"]),
        out_dir=out_dir,
        silhouette_index=1,
        sil_class=args["sil_class"],
        base_grid=int(args["base_grid"]),
        records=args["records"],
        classes=args["classes"],
        byClass=args["byClass"],
        n_images=int(args["n_attempts"]),
        rng=rng,
        start_index=int(args["start_index"]),
        fraction=float(args["fraction"]),
        final_n_samples_mode=args["final_n_samples_mode"],
        supersample=int(args["supersample"]),
        flush_every=int(args["flush_every"]),
        max_attempts_per_image=int(args["max_attempts_per_image"]),
        require_valid=True,
        snap_intersections_to_vertices=False,
        refit_enabled=False,
        refit_n_ctrl=int(args["refit_n_ctrl"]),
        refit_subdiv=int(args["refit_subdiv"]),
        refit_jitter_sigma=float(args["refit_jitter_sigma"]),
        refit_max_attempts=int(args["refit_max_attempts"]),
        shrink_gamma=float(args["shrink_gamma"]),
        max_shrink_iters=int(args["max_shrink_iters"]),
        smooth_win=int(args["smooth_win"]),
        try_mirror=True,
        save_invalid=False,
        invalid_subdir="_invalid",
    )

    meta_path = tmp_dir / f"meta_{wid:02d}.jsonl"
    save_metadata_jsonl(metas, meta_path)

    xy_path = tmp_dir / f"xy_{wid:02d}.npz"
    save_xy_npz(
        xy_path,
        out_files=out_files_xy,
        polygons=polygons_xy,
        base_grid=int(args["base_grid"]),
        matlab_1_indexed=True,
    )

    return str(meta_path), str(xy_path), fail_counts


def extract_idx(p: str) -> int:
    m = re.search(r"completion_\d{4}_(\d{5})\.png$", str(p))
    if not m:
        raise ValueError(f"Could not parse completion index from: {p}")
    return int(m.group(1))

In [12]:
print("Cases to run:")
display(
    run_df[["case_id", "model_type", "occluder_size", "target_logit_drop", "x0", "y0"]].head(20)
)

Cases to run:


,case_id,model_type,occluder_size,target_logit_drop,x0,y0
0,ns_cow_202,resnet50_geirhos_tl,8,0.385403,124,108
1,ns_cow_202,resnet50_tl_20250829,8,0.976028,60,176
2,ns_cow_205,resnet50_geirhos_tl,8,0.369282,72,40
3,ns_cow_205,resnet50_tl_20250829,8,1.358337,92,176
4,ns_cow_350,resnet50_geirhos_tl,8,0.418877,8,56
5,ns_cow_350,resnet50_tl_20250829,8,0.814728,48,48
6,ns_cow_510,resnet50_geirhos_tl,8,0.349799,20,76
7,ns_cow_510,resnet50_tl_20250829,8,0.993073,48,48
8,ns_cow_780,resnet50_geirhos_tl,8,0.329130,156,36
9,ns_cow_780,resnet50_tl_20250829,8,0.840988,156,172


In [13]:
# %%
def case_already_processed(out_dir: Path, require_nonempty: bool = True):
    """
    Returns True if the case looks already processed.

    Required files:
    - generated/shapes_meta.jsonl
    - generated/shapes_xy.npz
    - generated/source_occluder_config.json

    If require_nonempty=True, also require at least one completion PNG.
    """
    gen_dir = out_dir / "generated"

    meta_out = gen_dir / "shapes_meta.jsonl"
    xy_out = gen_dir / "shapes_xy.npz"
    config_out = gen_dir / "source_occluder_config.json"
    comp_dir = gen_dir / "completions"

    required_exist = meta_out.exists() and xy_out.exists() and config_out.exists()
    if not required_exist:
        return False

    if not require_nonempty:
        return True

    if not comp_dir.exists():
        return False

    has_png = any(comp_dir.glob("completion_*.png"))
    return has_png

# %%
def get_intersection_points_or_none(case: str, model: str, occ_size, silhouette, occluder):
    """
    Try to compute contour-occluder intersections.
    Returns the points array, or None if fewer than 2 points are found.
    """
    pts = find_intersection_points_multiple(silhouette, occluder, eps_merge=1e-3)

    if pts is None or pts.shape[0] < 2:
        print(
            f"SKIP {case} | {model} | size={occ_size}: "
            f"no usable intersection points found "
            f"(got {0 if pts is None else pts.shape[0]})."
        )
        return None

    return pts

In [14]:
# %%
print("Cases to run:")
display(
    run_df[["case_id", "model_type", "occluder_size", "target_logit_drop", "x0", "y0"]].head(20)
)

# %%
for row in run_df.to_dict(orient="records"):
    CASE = row["case_id"]
    MODEL = row["model_type"]
    OCC_SIZE = row["occluder_size"]

    print("\n=== CASE:", CASE, "| MODEL:", MODEL, "| SIZE:", OCC_SIZE)

    jsonl_path = find_case_jsonl(CASE, CASESET_DIR)

    OUT_DIR = OUT_PARENT / CASE / MODEL / f"size_{OCC_SIZE}"
    RAND_DIR = OUT_DIR / "generated" / "completions"
    TMP = OUT_DIR / "generated" / "_tmp"

    # Skip if already processed
    if case_already_processed(OUT_DIR, require_nonempty=True):
        print(f"SKIP {CASE} | {MODEL} | size={OCC_SIZE}: already processed.")
        continue

    RAND_DIR.mkdir(parents=True, exist_ok=True)
    TMP.mkdir(parents=True, exist_ok=True)

    sil_u, baseGrid, sil_class, base_row = load_case_from_jsonl(jsonl_path)

    silhouette = unit_to_pixel(sil_u, baseGrid)
    occluder_u, occluder = rect_poly_from_csv_row(pd.Series(row), baseGrid)

    polys = [silhouette, occluder]
    minX, minY, wBB, hBB = compute_bbox(
        polys,
        base_grid=baseGrid,
        margin=MARGIN
    )

    draw_and_save(
        polygons=[silhouette],
        colors=[[0, 0, 0]],
        minX=minX, minY=minY, wBB=wBB, hBB=hBB,
        out_w=W, out_h=H,
        out_file=OUT_DIR / "gt.png",
        supersample=SUPERSAMPLE
    )

    draw_and_save(
        polygons=[silhouette, occluder],
        colors=[[0, 0, 0], [131, 131, 131]],
        minX=minX, minY=minY, wBB=wBB, hBB=hBB,
        out_w=W, out_h=H,
        out_file=OUT_DIR / "occluded.png",
        supersample=SUPERSAMPLE
    )

    # Skip if no usable intersection points
    pts = get_intersection_points_or_none(
        CASE, MODEL, OCC_SIZE, silhouette, occluder
    )
    if pts is None:
        continue

    start_pt, end_pt, case_fraction = choose_gap_pair_and_fraction(
        silhouette.astype(float),
        occluder.astype(float),
        pts.astype(float),
    )

    CASE_MAX_ATTEMPTS = int(np.clip(
        250 + 2500 * abs(CASE_FRACTION - 0.22),
        250,
        4000
    ))

    print(
        f"{CASE} | {MODEL} | size={OCC_SIZE}: "
        f"fraction={CASE_FRACTION:.4f}, "
        f"derived_fraction={case_fraction:.4f}, "
        f"max_attempts={CASE_MAX_ATTEMPTS}"
    )

    case_seed = abs(hash((CASE, MODEL, int(OCC_SIZE)))) % (2**31 - 1)

    base_attempts = TOTAL_ATTEMPTS // N_WORKERS
    remainder = TOTAL_ATTEMPTS % N_WORKERS

    jobs = []
    next_start_index = 1

    for wid in range(N_WORKERS):
        n_attempts_w = base_attempts + (1 if wid < remainder else 0)
        worker_seed = int(case_seed + wid * 10_000 + 123)

        jobs.append(dict(
            worker_id=wid,
            seed=worker_seed,
            tmp_dir=str(TMP),
            out_dir=str(RAND_DIR),

            silhouette=silhouette,
            occluder=occluder,
            start_pt=start_pt,
            end_pt=end_pt,
            minX=minX,
            minY=minY,
            wBB=wBB,
            hBB=hBB,
            out_w=W,
            out_h=H,
            sil_class=sil_class,
            base_grid=baseGrid,

            records=records,
            classes=classes,
            byClass=byClass,

            n_attempts=n_attempts_w,
            start_index=next_start_index,

            fraction=CASE_FRACTION,
            final_n_samples_mode=SAMPLES_MODE,
            supersample=SUPERSAMPLE,
            flush_every=FLUSH,
            max_attempts_per_image=CASE_MAX_ATTEMPTS,

            refit_n_ctrl=REFIT_N_CTRL,
            refit_subdiv=REFIT_SUBDIV,
            refit_jitter_sigma=REFIT_JITTER,
            refit_max_attempts=REFIT_MAX_ATTEMPTS,

            shrink_gamma=CASE_SHRINK_GAMMA,
            max_shrink_iters=CASE_MAX_SHRINK_ITERS,
            smooth_win=CASE_SMOOTH_WIN,
        ))

        next_start_index += n_attempts_w

    assert next_start_index == TOTAL_ATTEMPTS + 1, (next_start_index, TOTAL_ATTEMPTS)

    results = []

    with ProcessPoolExecutor(max_workers=N_WORKERS) as ex:
        futures = [ex.submit(_worker_run, j) for j in jobs]
        for fut in as_completed(futures):
            results.append(fut.result(timeout=20))

    fail_counter = Counter()
    for _, _, fc in results:
        fail_counter.update(fc)

    print("AGGREGATED FAIL COUNTS:")
    for k, v in sorted(fail_counter.items()):
        print(f"  {k}: {v}")

    all_meta = []
    for m, _, _ in results:
        all_meta += Path(m).read_text(encoding="utf-8").splitlines()

    meta_rows = [json.loads(l) for l in all_meta if l.strip()]
    meta_rows.sort(key=lambda r: int(r.get("completion_index", 0)))

    for r in meta_rows:
        r["source_case_id"] = CASE
        r["source_model_type"] = MODEL
        r["source_occluder_size"] = OCC_SIZE
        r["source_target_logit_drop"] = float(row["target_logit_drop"])
        r["source_occluder_x0"] = float(row["x0"])
        r["source_occluder_y0"] = float(row["y0"])

    meta_out = OUT_DIR / "generated" / "shapes_meta.jsonl"
    meta_out.parent.mkdir(exist_ok=True, parents=True)

    with meta_out.open("w", encoding="utf-8") as f:
        for r in meta_rows:
            f.write(json.dumps(r) + "\n")

    pairs = []
    for _, xy, _ in results:
        npz = np.load(xy, allow_pickle=True)
        for fpath, poly in zip(npz["out_files"], npz["polygons"]):
            pairs.append((extract_idx(fpath), str(fpath), poly))

    pairs.sort(key=lambda t: t[0])

    out_files = [p[1] for p in pairs]
    polys_xy = [p[2] for p in pairs]

    xy_out = OUT_DIR / "generated" / "shapes_xy.npz"
    save_xy_npz(
        xy_out,
        out_files=out_files,
        polygons=polys_xy,
        base_grid=int(baseGrid),
        matlab_1_indexed=True,
    )

    config_out = OUT_DIR / "generated" / "source_occluder_config.json"
    config = {
        "case_id": CASE,
        "model_type": MODEL,
        "occluder_size": float(OCC_SIZE),
        "case_jsonl": str(jsonl_path),
        "best_occluder_csv": str(BEST_OCCLUDER_CSV),
        "x0": float(row["x0"]),
        "y0": float(row["y0"]),
        "x_center": float(row["x_center"]) if "x_center" in row else None,
        "y_center": float(row["y_center"]) if "y_center" in row else None,
        "x0_norm": float(row["x0_norm"]) if "x0_norm" in row else None,
        "y0_norm": float(row["y0_norm"]) if "y0_norm" in row else None,
        "x1_norm": float(row["x1_norm"]) if "x1_norm" in row else None,
        "y1_norm": float(row["y1_norm"]) if "y1_norm" in row else None,
        "target_logit_drop": float(row["target_logit_drop"]),
        "derived_hidden_fraction": float(case_fraction),
        "used_fraction": float(CASE_FRACTION),
        "n_intersection_points": int(pts.shape[0]),
        "start_pt": np.asarray(start_pt).tolist(),
        "end_pt": np.asarray(end_pt).tolist(),
    }
    config_out.write_text(json.dumps(config, indent=2))

    print("Saved:", CASE, MODEL, OCC_SIZE, "valid:", len(out_files))

Cases to run:


,case_id,model_type,occluder_size,target_logit_drop,x0,y0
0,ns_cow_202,resnet50_geirhos_tl,8,0.385403,124,108
1,ns_cow_202,resnet50_tl_20250829,8,0.976028,60,176
2,ns_cow_205,resnet50_geirhos_tl,8,0.369282,72,40
3,ns_cow_205,resnet50_tl_20250829,8,1.358337,92,176
4,ns_cow_350,resnet50_geirhos_tl,8,0.418877,8,56
5,ns_cow_350,resnet50_tl_20250829,8,0.814728,48,48
6,ns_cow_510,resnet50_geirhos_tl,8,0.349799,20,76
7,ns_cow_510,resnet50_tl_20250829,8,0.993073,48,48
8,ns_cow_780,resnet50_geirhos_tl,8,0.329130,156,36
9,ns_cow_780,resnet50_tl_20250829,8,0.840988,156,172



=== CASE: ns_cow_202 | MODEL: resnet50_geirhos_tl | SIZE: 8
SKIP ns_cow_202 | resnet50_geirhos_tl | size=8: already processed.

=== CASE: ns_cow_202 | MODEL: resnet50_tl_20250829 | SIZE: 8
SKIP ns_cow_202 | resnet50_tl_20250829 | size=8: already processed.

=== CASE: ns_cow_205 | MODEL: resnet50_geirhos_tl | SIZE: 8
SKIP ns_cow_205 | resnet50_geirhos_tl | size=8: already processed.

=== CASE: ns_cow_205 | MODEL: resnet50_tl_20250829 | SIZE: 8
SKIP ns_cow_205 | resnet50_tl_20250829 | size=8: already processed.

=== CASE: ns_cow_350 | MODEL: resnet50_geirhos_tl | SIZE: 8
SKIP ns_cow_350 | resnet50_geirhos_tl | size=8: already processed.

=== CASE: ns_cow_350 | MODEL: resnet50_tl_20250829 | SIZE: 8
SKIP ns_cow_350 | resnet50_tl_20250829 | size=8: already processed.

=== CASE: ns_cow_510 | MODEL: resnet50_geirhos_tl | SIZE: 8
SKIP ns_cow_510 | resnet50_geirhos_tl | size=8: already processed.

=== CASE: ns_cow_510 | MODEL: resnet50_tl_20250829 | SIZE: 8
SKIP ns_cow_510 | resnet50_tl_202508

## Notes

This version changes only the occluder source.

### Old source
- occluder taken from the base JSONL

### New source
- occluder taken from the best-on-contour CSV
- silhouette and class metadata still taken from the base JSONL

### Important consequence
The recomputed intersections, hidden arc, and derived fraction are now tied to the **selected best occluder**, not to the original occluder stored in the case JSONL.